[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/02_dit/02_dit.ipynb)

# 02 · Diffusion Transformer（用 numpy 玩具复现）

目标：用纯 numpy 把 **DiT 的每个零件**从零搭出来：① patchify/unpatchify 把小图序列化；② 正弦时间步嵌入；③ **adaLN-zero** 条件注入（最关键）；④ 一个完整 DiT block 前向。

路线：patchify → unpatchify 往返 → 2D 位置嵌入 → 正弦时间步嵌入 → adaLN 调制 → adaLN-zero 恒等性质 → 一个 DiT block → ✏️ 练习 → 📖 答案 → 🧪 真实 DiT 配置胶囊。

> 心智模型：**DiT = 把图切成块当 token，用 Transformer 处理，adaLN 注入时间步/条件**。注意力是玩具实现，但 patchify/adaLN-zero/时间步嵌入的逻辑与真实 DiT 逐行对应。

## 1 · Patchify：把 latent 切成 token 序列

把 `H×W×C` 的 latent 切成 `p×p` 的不重叠块、每块展平投影成一个 token。token 数 `n=(H/p)·(W/p)`。

先实现切块 + 展平（投影留到后面）：返回 `(n, p*p*C)` 的 patch 矩阵。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def patchify(x, p):
    '''x: (H,W,C) latent。切成 p×p 块、每块展平。返回 (n, p*p*C)。'''
    H, W, C = x.shape
    assert H % p == 0 and W % p == 0, 'H,W 必须被 p 整除'
    nh, nw = H // p, W // p
    patches = []
    for i in range(nh):
        for j in range(nw):
            block = x[i*p:(i+1)*p, j*p:(j+1)*p, :]   # (p,p,C)
            patches.append(block.reshape(-1))         # p*p*C
    return np.stack(patches, 0)                       # (n, p*p*C)

H, W, C, p = 8, 8, 3, 2
x = rng.standard_normal((H, W, C))
patches = patchify(x, p)
n_expected = (H // p) * (W // p)
print(f'latent {H}x{W}x{C}, patch={p} -> {patches.shape[0]} 个 token, 每个 {patches.shape[1]} 维')
assert patches.shape == (n_expected, p * p * C)
assert n_expected == 16 and patches.shape[1] == 12
print('✅ patchify：8x8x3 latent 在 p=2 下切成 16 个 token，每 token 12 维')

## 2 · Unpatchify：拼回 latent（往返无损）

输出端的逆操作：把 token 矩阵拼回 `H×W×C`。验证 **`unpatchify(patchify(x)) == x`**（往返无损是 DiT 形状一致的保证）。

In [ ]:
def unpatchify(patches, H, W, C, p):
    '''patches: (n, p*p*C) -> 拼回 (H,W,C)。'''
    nh, nw = H // p, W // p
    x = np.zeros((H, W, C))
    k = 0
    for i in range(nh):
        for j in range(nw):
            x[i*p:(i+1)*p, j*p:(j+1)*p, :] = patches[k].reshape(p, p, C)
            k += 1
    return x

x_rec = unpatchify(patchify(x, p), H, W, C, p)
assert x_rec.shape == x.shape
assert np.allclose(x_rec, x), 'patchify/unpatchify 往返必须无损'
print('✅ unpatchify(patchify(x)) == x —— DiT 进出形状一致的保证')
# 不同 patch 大小都应往返无损
for pp in [1, 2, 4]:
    assert np.allclose(unpatchify(patchify(x, pp), H, W, C, pp), x)
print('   p=1,2,4 往返均无损 ✅')

## 3 · Patch 嵌入 + 2D 位置嵌入

把每个展平 patch 线性投影到隐藏维 `d`，再加 **2D 位置嵌入**（注意力无空间概念，必须显式注入位置）。

位置嵌入用每个 patch 的 `(行, 列)` 坐标做正弦编码。验证：加位置后，**不同位置的相同 patch 得到不同 token**。

In [ ]:
d = 16   # 隐藏维
Wproj = rng.standard_normal((p * p * C, d)) * 0.1   # patch 嵌入投影

def sincos_2d(nh, nw, d):
    '''每个 (i,j) patch 的 2D 正弦位置嵌入，返回 (nh*nw, d)。'''
    assert d % 4 == 0
    dq = d // 4
    div = 10000 ** (np.arange(dq) / dq)
    pos = []
    for i in range(nh):
        for j in range(nw):
            row = np.concatenate([np.sin(i/div), np.cos(i/div)])
            col = np.concatenate([np.sin(j/div), np.cos(j/div)])
            pos.append(np.concatenate([row, col]))
    return np.stack(pos, 0)

nh, nw = H // p, W // p
tokens = patchify(x, p) @ Wproj            # (n, d)
pos_emb = sincos_2d(nh, nw, d)             # (n, d)
tokens_pe = tokens + pos_emb
print('tokens', tokens.shape, ' pos_emb', pos_emb.shape)
assert tokens_pe.shape == (nh * nw, d)
# 位置嵌入应让相同内容在不同位置区分开
same_patch = np.ones(p*p*C) @ Wproj
tok_at_0 = same_patch + pos_emb[0]
tok_at_5 = same_patch + pos_emb[5]
assert not np.allclose(tok_at_0, tok_at_5), '同内容不同位置应得到不同 token'
# 不同位置的位置嵌入确实不同
assert not np.allclose(pos_emb[0], pos_emb[1])
print('✅ patch 嵌入 + 2D 位置嵌入：同 patch 在不同位置被区分（空间结构注入）')

## 4 · 正弦时间步嵌入

把标量时间步 `t` 用多频 sin/cos 编码成向量，让网络知道「现在多噪」。

验证：① 不同 `t` 得到不同嵌入；② 相近的 `t` 嵌入也相近（**平滑**，这是正弦编码的关键好处）。

In [ ]:
def timestep_embedding(t, dim, max_period=10000):
    '''t: 标量或 (B,) 数组。返回 (..., dim) 正弦嵌入。'''
    t = np.atleast_1d(np.asarray(t, dtype=float))
    half = dim // 2
    freqs = np.exp(-np.log(max_period) * np.arange(half) / half)
    args = t[:, None] * freqs[None, :]
    emb = np.concatenate([np.cos(args), np.sin(args)], axis=-1)
    return emb

emb = timestep_embedding(np.array([0, 1, 2, 50, 999]), dim=16)
print('时间步嵌入 shape:', emb.shape)
assert emb.shape == (5, 16)
# 不同 t 嵌入不同
assert not np.allclose(emb[0], emb[3])
# 平滑性：相邻 t 的嵌入距离 < 远隔 t 的嵌入距离
d_close = np.linalg.norm(timestep_embedding(100,16) - timestep_embedding(101,16))
d_far   = np.linalg.norm(timestep_embedding(100,16) - timestep_embedding(900,16))
print(f'|emb(100)-emb(101)| = {d_close:.3f}   |emb(100)-emb(900)| = {d_far:.3f}')
assert d_close < d_far, '相近时间步的嵌入应更接近（平滑）'
print('✅ 正弦时间步嵌入：不同 t 可区分、相近 t 平滑 —— 给网络一个连续的「闹钟」')

## 5 · adaLN 调制：用条件预测 γ,β,α

AdaLN 把 LayerNorm 的固定 `γ,β` 换成**由条件向量预测**的。DiT 每个 block 还预测残差缩放 `α`。

实现：条件向量 `c` -> 线性层 -> `(γ, β, α)`。先看普通初始化（非零）下，adaLN 确实改变了归一化行为。

In [ ]:
def layernorm(h, eps=1e-5):
    mu = h.mean(-1, keepdims=True)
    var = h.var(-1, keepdims=True)
    return (h - mu) / np.sqrt(var + eps)

def make_adaln(d, zero_init=False):
    '''返回一个把条件 c (d,) 映到 (gamma,beta,alpha) 各 (d,) 的线性调制器。'''
    scale = 0.0 if zero_init else 0.1
    W = rng.standard_normal((d, 3 * d)) * scale
    b = np.zeros(3 * d)
    def modulate(c):
        out = c @ W + b              # (3d,)
        gamma, beta, alpha = out[:d], out[d:2*d], out[2*d:]
        return gamma, beta, alpha
    return modulate

cond = timestep_embedding(50, d)[0]    # 条件向量 = 时间步嵌入
mod = make_adaln(d, zero_init=False)
gamma, beta, alpha = mod(cond)
h = rng.standard_normal((4, d))
h_norm = layernorm(h)
h_mod = (1 + gamma) * h_norm + beta     # DiT 用 (1+γ) 使初始为标准 LN
print('γ,β,α shapes:', gamma.shape, beta.shape, alpha.shape)
assert gamma.shape == (d,) and alpha.shape == (d,)
assert not np.allclose(h_mod, h_norm), '非零 adaLN 应改变归一化输出'
print('✅ adaLN：条件向量预测 γ,β,α，动态调制每层归一化（FiLM 思想）')

## 6 · adaLN-zero：初始时 block 是恒等映射（最关键）

DiT 的画龙点睛：把残差缩放 `α` 的预测层**零初始化**，使训练开始时 `α=0`，于是 `h ← h + α·sublayer(...) = h`——**每个 block 初始就是恒等映射**。

这让极深 DiT 能稳定训练。验证：zero-init 时，整个 block 的输出 == 输入。

In [ ]:
def dit_block_forward(h, cond, mod, attn_fn, mlp_fn):
    '''一个 DiT block：adaLN 调制 -> 子层 -> α 缩放残差。'''
    g1, b1, a1 = mod(cond)      # 这里玩具用同一组调制给两个子层(真实 DiT 预测 6 组)
    # 注意力子层
    h = h + a1 * attn_fn((1 + g1) * layernorm(h) + b1)
    g2, b2, a2 = mod(cond)
    h = h + a2 * mlp_fn((1 + g2) * layernorm(h) + b2)
    return h

# 玩具子层（内容不重要，关键看 zero-init 时被 α=0 抹掉）
Wa = rng.standard_normal((d, d)) * 0.5
Wm = rng.standard_normal((d, d)) * 0.5
attn_fn = lambda z: z @ Wa
mlp_fn = lambda z: np.maximum(z @ Wm, 0)

h0 = rng.standard_normal((4, d))
# zero-init adaLN：α=0 -> block 恒等
mod_zero = make_adaln(d, zero_init=True)
h_out_zero = dit_block_forward(h0, cond, mod_zero, attn_fn, mlp_fn)
print('zero-init: |block(h)-h| =', np.abs(h_out_zero - h0).max())
assert np.allclose(h_out_zero, h0), 'adaLN-zero 时 block 必须是恒等映射'
# 普通 init：block 会改变输入
mod_nz = make_adaln(d, zero_init=False)
h_out_nz = dit_block_forward(h0, cond, mod_nz, attn_fn, mlp_fn)
assert not np.allclose(h_out_nz, h0)
print('✅ adaLN-zero：α=0 使每个 block 从恒等起步 —— DiT 能 scale 到极深的关键')

---
## ✏️ 练习 1：patchify 的 token 数与维度

实现 `patch_shapes(H, W, C, p)`：返回 `(token 数 n, 每 token 维度)`，不实际切块、纯算形状。

并回答：latent `32×32×4`、`p=2` 时序列多长？（这关系到注意力的 `O(n²)` 成本。）

In [ ]:
def patch_shapes(H, W, C, p):
    # TODO: 返回 (n, dim) = ((H/p)*(W/p), p*p*C)，要求 H,W 被 p 整除
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert patch_shapes(8, 8, 3, 2) == (16, 12)
assert patch_shapes(32, 32, 4, 2) == (256, 16)   # SD3 量级
assert patch_shapes(32, 32, 4, 4) == (64, 64)    # p 翻倍 -> token 降 4 倍
# p 越大 token 越少（注意力越便宜）
n2 = patch_shapes(32, 32, 4, 2)[0]
n4 = patch_shapes(32, 32, 4, 4)[0]
assert n2 == 4 * n4, 'p 翻倍，token 数降为 1/4，注意力成本降 1/16'
print('32x32x4 latent: p=2 -> 256 tokens, p=4 -> 64 tokens')
print('✅ 练习 1 通过：patch 大小是序列长度（=算力）的核心旋钮')

## ✏️ 练习 2：adaLN 调制函数

实现 `apply_adaln(h_norm, gamma, beta)`：对已归一化的 `h_norm` 做 `(1+gamma)*h_norm + beta`（DiT 用 `1+γ` 使初始为标准 LN）。

并实现 `modulated_residual(h, sublayer_out, alpha)`：残差 `h + alpha*sublayer_out`。

In [ ]:
def apply_adaln(h_norm, gamma, beta):
    # TODO: (1+gamma)*h_norm + beta
    raise NotImplementedError

def modulated_residual(h, sublayer_out, alpha):
    # TODO: h + alpha*sublayer_out
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
hn = rng.standard_normal((3, d))
# gamma=0,beta=0 -> 恒等（标准 LN，因为用 1+gamma）
assert np.allclose(apply_adaln(hn, np.zeros(d), np.zeros(d)), hn)
# gamma=1 -> 放大 2 倍
assert np.allclose(apply_adaln(hn, np.ones(d), np.zeros(d)), 2 * hn)
# alpha=0 -> 残差不变（adaLN-zero 的本质）
so = rng.standard_normal((3, d))
assert np.allclose(modulated_residual(hn, so, 0.0), hn)
assert np.allclose(modulated_residual(hn, so, 1.0), hn + so)
print('✅ 练习 2 通过：γ=β=0 时 adaLN 为恒等、α=0 时残差为恒等（zero-init 基础）')

## ✏️ 练习 3：时间步嵌入的局部平滑性

实现 `embedding_distance(t1, t2, dim)`：返回两个时间步嵌入的欧氏距离。

用它验证正弦编码的关键性质：**相邻时间步的嵌入很接近**（局部平滑），而**相隔很远的时间步嵌入明显更远**。（注意：正弦编码含多个频率、整体并非严格单调，但「近 ≪ 远」这条在小邻域内稳健成立。）

In [ ]:
def embedding_distance(t1, t2, dim):
    # TODO: 用 timestep_embedding 算两个嵌入，返回欧氏距离
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(embedding_distance(500, 500, 16)) < 1e-9, '同 t 距离为 0'
# 局部平滑：Δ=1 的距离应远小于 Δ=400 的距离
d1   = embedding_distance(500, 501, 16)
d400 = embedding_distance(500, 900, 16)
print(f'|emb(500)-emb(501)| = {d1:.4f}   |emb(500)-emb(900)| = {d400:.4f}')
assert d1 < d400, '相邻时间步应比远隔时间步接近得多'
# 在很小的邻域内（Δ=1,2,3）距离应随 Δ 增大（低频主导、尚未进入振荡）
small = [embedding_distance(500, 500 + k, 16) for k in range(0, 4)]
assert all(small[i] <= small[i+1] + 1e-9 for i in range(len(small)-1)), '小邻域内应单调'
print('小邻域距离(Δ=0,1,2,3):', np.round(small, 4))
print('✅ 练习 3 通过：时间步嵌入局部平滑，网络能感知噪声级别的连续性')

## ✏️ 练习 4：序列化往返保形

实现 `roundtrip_ok(x, p)`：对 latent `x` 做 patchify -> unpatchify，返回是否形状与数值都还原（bool）。

验证它对多种 `(H,W,C,p)` 组合都为 True，并对 `H` 不被 `p` 整除时正确报错。

In [ ]:
def roundtrip_ok(x, p):
    # TODO: H,W,C = x.shape; rec = unpatchify(patchify(x,p),H,W,C,p)
    #       返回 rec.shape==x.shape and np.allclose(rec,x)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
for (HH, WW, CC, pp) in [(8,8,3,2), (8,8,3,4), (6,6,2,2), (4,4,1,1)]:
    xt = rng.standard_normal((HH, WW, CC))
    assert roundtrip_ok(xt, pp), f'往返应无损 @ {(HH,WW,CC,pp)}'
# H 不被 p 整除应报错
bad = rng.standard_normal((7, 8, 3))
try:
    patchify(bad, 2)
    raised = False
except AssertionError:
    raised = True
assert raised, 'H=7 不被 p=2 整除应触发断言'
print('✅ 练习 4 通过：序列化往返保形，且非整除时正确报错')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def patch_shapes(H, W, C, p):
    assert H % p == 0 and W % p == 0
    return ((H // p) * (W // p), p * p * C)

In [ ]:
# 练习 2 参考答案
def apply_adaln(h_norm, gamma, beta):
    return (1 + gamma) * h_norm + beta
def modulated_residual(h, sublayer_out, alpha):
    return h + alpha * sublayer_out

In [ ]:
# 练习 3 参考答案
def embedding_distance(t1, t2, dim):
    e1 = timestep_embedding(t1, dim)[0]
    e2 = timestep_embedding(t2, dim)[0]
    return float(np.linalg.norm(e1 - e2))

In [ ]:
# 练习 4 参考答案
def roundtrip_ok(x, p):
    H, W, C = x.shape
    rec = unpatchify(patchify(x, p), H, W, C, p)
    return rec.shape == x.shape and bool(np.allclose(rec, x))

---
## 🧪 真实数据胶囊：DiT 配置与算力

用**真实 DiT 模型族**的配置，算 token 数与注意力成本。下面是 Peebles & Xie 2023 的 DiT 配置（在 256×256 ImageNet、f=8 VAE -> 32×32×4 latent 上）。

DiT-S/B/L/XL 对应不同深度/宽度，patch p∈{2,4,8} 对应不同 token 数。算每个配置的序列长度与注意力 FLOPs 量级。

In [ ]:
# 真实 DiT 配置（Peebles & Xie 2023, ImageNet 256, latent 32x32x4）
DIT_MODELS = {
    'DiT-S': dict(depth=12, hidden=384,  heads=6),
    'DiT-B': dict(depth=12, hidden=768,  heads=12),
    'DiT-L': dict(depth=24, hidden=1024, heads=16),
    'DiT-XL':dict(depth=28, hidden=1152, heads=16),
}
LATENT = (32, 32, 4)

def n_tokens(latent, p):
    H, W, _ = latent
    return (H // p) * (W // p)

print('patch p 对 token 数(注意力序列长度)的影响 (latent 32x32):')
for p in [2, 4, 8]:
    n = n_tokens(LATENT, p)
    print(f'  p={p}: {n:3d} tokens  -> 注意力∝n² = {n*n:,}')
print()
# 注意力 FLOPs 量级 ∝ depth * n² * hidden（粗略）
p = 2
n = n_tokens(LATENT, p)
print(f'各 DiT (p={p}, n={n} tokens) 的注意力计算量级 (∝ depth·n²·hidden):')
for name, cfg in DIT_MODELS.items():
    flops = cfg['depth'] * n * n * cfg['hidden']
    print(f"  {name:7s} depth={cfg['depth']:2d} hidden={cfg['hidden']:4d} -> {flops/1e6:8.1f}M (相对)")
print('\n观察：DiT-XL 比 DiT-S 计算量大~10x，FID 也显著更低 —— 这就是 DiT 的 scaling law')

**🧪 胶囊练习**：实现 `attention_cost_ratio(latent, p1, p2)`：返回 patch `p1` 相对 `p2` 的注意力成本倍数（∝ token 数平方之比）。

并算：把 patch 从 4 改成 2，注意力成本涨几倍？

In [ ]:
def attention_cost_ratio(latent, p1, p2):
    # TODO: 返回 (n_tokens(latent,p1)/n_tokens(latent,p2))**2
    raise NotImplementedError

In [ ]:
# 自测
# p=2 vs p=4: token 数 4 倍 -> 注意力成本 16 倍
r = attention_cost_ratio((32, 32, 4), 2, 4)
assert abs(r - 16.0) < 1e-6, 'p 减半 token×4 -> 注意力×16'
# p=4 vs p=8 同理
assert abs(attention_cost_ratio((32,32,4), 4, 8) - 16.0) < 1e-6
# 同 patch 比值为 1
assert abs(attention_cost_ratio((32,32,4), 2, 2) - 1.0) < 1e-6
print('把 patch 从 4 改成 2：注意力成本涨 16 倍（token 数平方）')
print('✅ 胶囊练习通过：理解 patch 大小如何主导 DiT 的注意力算力')

In [ ]:
# 📖 胶囊参考答案
def attention_cost_ratio(latent, p1, p2):
    return (n_tokens(latent, p1) / n_tokens(latent, p2)) ** 2

---
### 小结
- **DiT = 用 Transformer 当扩散去噪骨干**：patchify 把 latent 切成 token 序列，N 层 Transformer 处理，unpatchify 拼回。
- **patchify/unpatchify** 往返无损；patch 大小 `p` 决定 token 数 `n=(H/p)(W/p)`，从而决定注意力的 `O(n²)` 成本（核心算力旋钮）。
- **时间步嵌入**用正弦编码把标量 t 平滑展开成向量，告诉网络当前噪声级别；条件向量 = 时间步 + 类别/文本。
- **adaLN-zero（最关键）**：条件预测 γ,β,α 调制每层归一化与残差；α 零初始化使每个 block 从**恒等**起步，让极深 DiT 稳定可训。
- **DiT vs U-Net**：弱归纳偏置但 scaling 平滑、长程依赖强；与 latent 化相互成就，是 SD3/Flux/SoRA 的共同骨干。

下一站：**模块 03 · Flow Matching** —— 把「扩散」这个学习目标本身升级成更直接的「学速度场、积分 ODE」。